# repocoder-mine — full run with StarCoder-7B

Clones this repo (code + the pre-built `data/repositories/` snippet database + vendored `cceval/`), loads `bigcode/starcoderbase-7b` locally via `transformers`, and runs `run_completion.py --backend huggingface` over the CCEval examples.

**Runtime**: Runtime > Change runtime type > GPU (A100/L4 recommended — a 7B model in fp16 needs ~14GB VRAM; T4 works but is slower).

## 1. Clone the repo

In [ ]:
!git clone https://github.com/Robertkiza0/repocoder-mine.git
%cd repocoder-mine

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt torch transformers accelerate

## 3. Hugging Face access

`bigcode/starcoderbase-7b` is gated: accept the license at
https://huggingface.co/bigcode/starcoderbase-7b, then log in below with a
read token (masked input, never written to disk).

In [ ]:
from huggingface_hub import login
from getpass import getpass

login(token=getpass("HF token: "))

## 4. Get `line_completion.jsonl`

Not included in the repo (CCEval's raw dataset) — upload it here.

In [ ]:
from google.colab import files

uploaded = files.upload()  # select line_completion.jsonl
LINE_COMPLETION_PATH = next(iter(uploaded))

## 5. Run the experiment (50 examples)

Evaluates 50 examples from `line_completion.jsonl`. Each example runs 2 RepoCoder iterations, so this is 100 generations on a 7B model — change `SAMPLE` below (`0` = the full dataset) if needed.

In [ ]:
SAMPLE = 50  # number of examples to evaluate; 0 = full dataset

!python run_completion.py -o results/starcoder7b \
    --backend huggingface \
    --model bigcode/starcoderbase-7b \
    --line-completion-path "{LINE_COMPLETION_PATH}" \
    --sample {SAMPLE}

## 6. Inspect results

In [ ]:
import json

with open("results/starcoder7b/metrics.json", encoding="utf-8") as f:
    print(json.dumps(json.load(f), indent=2))

with open("results/starcoder7b/results.jsonl", encoding="utf-8") as f:
    results = [json.loads(line) for line in f]
print(f"\n{len(results)} completions saved.")
for r in results[:5]:
    print(r["task_id"], "exact_match=", r["exact_match"], "edit_similarity=", r["edit_similarity"])

## 7. Download the results

Zips `results/starcoder7b/` (results.jsonl, metrics.json, args.json) for download — the repo's own git history isn't touched from here.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("starcoder7b_results", "zip", "results/starcoder7b")
files.download("starcoder7b_results.zip")